In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import ViTMAEModel, AutoFeatureExtractor
from tqdm import tqdm
import numpy as np

# ── Config ──────────────────────────────────────────────────────────────────
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME  = "facebook/vit-mae-base"
BATCH_SIZE  = 256
EPOCHS      = 30
LR          = 1e-3
FEAT_DIM    = 768          # vit-mae-base hidden size
NUM_CLASSES = 10

print(f"Using device: {DEVICE}")



# ── Extract features (cache to RAM — faster than recomputing every epoch) ──





Using device: cuda


In [2]:
# ── Data ─────────────────────────────────────────────────────────────────────
# MAE ViT expects 224×224; CIFAR-10 is 32×32 so we resize
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ── Load pretrained MAE encoder (frozen) ─────────────────────────────────────
print(f"Loading {MODEL_NAME} ...")
mae = ViTMAEModel.from_pretrained(MODEL_NAME)
mae = mae.to(DEVICE)
mae.eval()
for p in mae.parameters():
    p.requires_grad_(False)

100%|██████████| 170M/170M [51:01<00:00, 55.7kB/s]


Loading facebook/vit-mae-base ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/448M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTMAEModel LOAD REPORT from: facebook/vit-mae-base
Key                                                                     | Status     |  | 
------------------------------------------------------------------------+------------+--+-
decoder.decoder_norm.weight                                             | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_before.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.q_proj.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.v_proj.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.o_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.k_proj.bias   | UNEXPECTED |  | 
decoder.decoder_embed.bias                                              | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.mlp.fc2.bias            | UNEXPECTED |  | 
decoder.decoder_norm.bi

In [3]:
def extract_features(loader, desc="Extracting"):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc=desc):
            imgs = imgs.to(DEVICE)
            # ViTMAEModel forward: outputs.last_hidden_state shape [B, 1+N_patches, D]
            # [CLS] token is index 0
            out = mae(pixel_values=imgs)
            cls = out.last_hidden_state[:, 0, :]   # [B, 768]
            feats.append(cls.cpu())
            labels.append(lbls)
    return torch.cat(feats), torch.cat(labels)


In [4]:
print("Extracting train features ...")
train_feats, train_labels = extract_features(train_loader, "Train")
print("Extracting test features ...")
test_feats,  test_labels  = extract_features(test_loader,  "Test")

# Move to device for training
train_feats  = train_feats.to(DEVICE)
train_labels = train_labels.to(DEVICE)
test_feats   = test_feats.to(DEVICE)
test_labels  = test_labels.to(DEVICE)

# ── Linear probe ──────────────────────────────────────────────────────────────
probe = nn.Linear(FEAT_DIM, NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(probe.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

# Use TensorDataset so we don't hit disk each step
from torch.utils.data import TensorDataset
probe_train_loader = DataLoader(
    TensorDataset(train_feats, train_labels),
    batch_size=512, shuffle=True
)

best_acc = 0.0
history  = {"train_loss": [], "test_acc": []}

print("\nTraining linear probe ...")

Extracting train features ...


Train: 100%|██████████| 196/196 [02:44<00:00,  1.19it/s]


Extracting test features ...


Test: 100%|██████████| 40/40 [00:36<00:00,  1.11it/s]


Training linear probe ...


In [5]:
for epoch in range(1, EPOCHS + 1):
    probe.train()
    total_loss = 0.0
    for xb, yb in probe_train_loader:
        optimizer.zero_grad()
        loss = criterion(probe(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    scheduler.step()

    # Eval
    probe.eval()
    with torch.no_grad():
        logits = probe(test_feats)
        preds  = logits.argmax(dim=1)
        acc    = (preds == test_labels).float().mean().item() * 100

    avg_loss = total_loss / len(train_ds)
    history["train_loss"].append(avg_loss)
    history["test_acc"].append(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(probe.state_dict(), "mae_probe_best.pt")

    print(f"Epoch {epoch:3d}/{EPOCHS} | loss {avg_loss:.4f} | test acc {acc:.2f}%")

print(f"\nBest test accuracy: {best_acc:.2f}%")

# ── Save results ──────────────────────────────────────────────────────────────
np.save("mae_probe_history.npy", history)
print("Saved history to mae_probe_history.npy")
print("Saved best probe weights to mae_probe_best.pt")

Epoch   1/30 | loss 1.8772 | test acc 64.43%
Epoch   2/30 | loss 1.3782 | test acc 68.00%
Epoch   3/30 | loss 1.1578 | test acc 70.41%
Epoch   4/30 | loss 1.0352 | test acc 71.85%
Epoch   5/30 | loss 0.9558 | test acc 72.95%
Epoch   6/30 | loss 0.8998 | test acc 73.93%
Epoch   7/30 | loss 0.8569 | test acc 74.93%
Epoch   8/30 | loss 0.8241 | test acc 75.53%
Epoch   9/30 | loss 0.7978 | test acc 75.87%
Epoch  10/30 | loss 0.7757 | test acc 76.01%
Epoch  11/30 | loss 0.7578 | test acc 76.53%
Epoch  12/30 | loss 0.7429 | test acc 76.70%
Epoch  13/30 | loss 0.7298 | test acc 77.10%
Epoch  14/30 | loss 0.7187 | test acc 77.40%
Epoch  15/30 | loss 0.7091 | test acc 77.43%
Epoch  16/30 | loss 0.7011 | test acc 77.64%
Epoch  17/30 | loss 0.6941 | test acc 77.73%
Epoch  18/30 | loss 0.6881 | test acc 77.90%
Epoch  19/30 | loss 0.6831 | test acc 77.92%
Epoch  20/30 | loss 0.6790 | test acc 78.11%
Epoch  21/30 | loss 0.6756 | test acc 78.02%
Epoch  22/30 | loss 0.6725 | test acc 78.12%
Epoch  23/

In [11]:
from google.colab import drive
drive.mount('/content/drive')

# First time only — downloads and saves to your Drive

mae.save_pretrained("/content/drive/MyDrive/mae_base")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
!pip install google-colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 91.9 MB/s eta 0:00:00
